# Phase 1 Tree Models

In [ ]:
from pathlib import Path
import gc

import lightgbm as lgb
import numpy as np
import pandas as pd

try:
    import xgboost as xgb
except ImportError:
    xgb = None

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / "data" / "top20_p8"
LGBM_MODEL_DIR = PROJECT_ROOT / "models" / "lgbm" / "top20_lgbm"

TARGET_COL = "responder_6"
WEIGHT_COL = "weight"
DATE_COL = "date_id"
CATEGORICAL_COLS = ["symbol_id", "feature_11"]
SEED = 42
SAMPLE_TRAIN_ROWS = 300_000
SAMPLE_VALID_ROWS = 120_000

print({"xgboost_available": xgb is not None})

In [ ]:
def weighted_r2(y_true, y_pred, weight):
    y_true = np.asarray(y_true, dtype=np.float64)
    y_pred = np.asarray(y_pred, dtype=np.float64)
    weight = np.asarray(weight, dtype=np.float64)
    numerator = np.sum(weight * np.square(y_true - y_pred))
    denominator = np.sum(weight * np.square(y_true))
    return np.nan if denominator == 0 else 1.0 - numerator / denominator


def load_split(name):
    df = pd.read_parquet(DATA_DIR / f"{name}.parquet")
    for col in CATEGORICAL_COLS:
        if col in df.columns:
            df[col] = df[col].astype("category")
    return df


def feature_cols(df):
    return [col for col in df.columns if col not in [DATE_COL, WEIGHT_COL, TARGET_COL]]


def sample_rows(df, n):
    if len(df) <= n:
        return df.copy()
    return df.sample(n=n, random_state=SEED).sort_values([DATE_COL, "time_id", "symbol_id"]).reset_index(drop=True)

In [ ]:
train_df = sample_rows(load_split("train"), SAMPLE_TRAIN_ROWS)
valid_df = sample_rows(load_split("valid"), SAMPLE_VALID_ROWS)
features = feature_cols(train_df)
categorical_features = [col for col in CATEGORICAL_COLS if col in features]

print({
    "train": train_df.shape,
    "valid": valid_df.shape,
    "features": len(features),
    "categorical_features": categorical_features,
    "train_dates": (int(train_df[DATE_COL].min()), int(train_df[DATE_COL].max())),
    "valid_dates": (int(valid_df[DATE_COL].min()), int(valid_df[DATE_COL].max())),
})

In [ ]:
def train_lgbm(train_df, valid_df, features, params_override=None, rounds=500):
    params = {
        "objective": "regression",
        "metric": "rmse",
        "boosting_type": "gbdt",
        "device_type": "cpu",
        "num_leaves": 64,
        "learning_rate": 0.03,
        "feature_fraction": 0.9,
        "bagging_fraction": 0.9,
        "bagging_freq": 1,
        "min_data_in_leaf": 500,
        "lambda_l2": 1.0,
        "seed": SEED,
        "feature_pre_filter": False,
        "verbosity": -1,
    }
    if params_override:
        params.update(params_override)

    train_set = lgb.Dataset(
        train_df[features],
        label=train_df[TARGET_COL],
        weight=train_df[WEIGHT_COL],
        feature_name=features,
        categorical_feature=[col for col in categorical_features if col in features],
        free_raw_data=False,
    )
    valid_set = lgb.Dataset(
        valid_df[features],
        label=valid_df[TARGET_COL],
        weight=valid_df[WEIGHT_COL],
        feature_name=features,
        categorical_feature=[col for col in categorical_features if col in features],
        free_raw_data=False,
    )
    model = lgb.train(
        params,
        train_set,
        valid_sets=[train_set, valid_set],
        valid_names=["train", "valid"],
        num_boost_round=rounds,
        callbacks=[lgb.early_stopping(50), lgb.log_evaluation(50)],
    )
    pred = model.predict(valid_df[features], num_iteration=model.best_iteration)
    return model, weighted_r2(valid_df[TARGET_COL], pred, valid_df[WEIGHT_COL])


lgbm_model, lgbm_score = train_lgbm(train_df, valid_df, features)
print(f"sample_lgbm_valid_weighted_r2={lgbm_score:.8f}")

In [ ]:
if xgb is None:
    print("xgboost is not installed locally; run the Longleaf script for the full XGBoost check.")
else:
    xgb_train = train_df.copy()
    xgb_valid = valid_df.copy()
    for col in categorical_features:
        xgb_train[col] = xgb_train[col].cat.codes.astype("int16")
        xgb_valid[col] = xgb_valid[col].cat.codes.astype("int16")

    xgb_model = xgb.XGBRegressor(
        n_estimators=500,
        max_depth=5,
        learning_rate=0.03,
        subsample=0.9,
        colsample_bytree=0.9,
        min_child_weight=100,
        reg_lambda=1.0,
        objective="reg:squarederror",
        tree_method="hist",
        device="cpu",
        random_state=SEED,
        eval_metric="rmse",
    )
    xgb_model.fit(
        xgb_train[features],
        xgb_train[TARGET_COL],
        sample_weight=xgb_train[WEIGHT_COL],
        eval_set=[(xgb_valid[features], xgb_valid[TARGET_COL])],
        sample_weight_eval_set=[xgb_valid[WEIGHT_COL]],
        verbose=50,
    )
    xgb_pred = xgb_model.predict(xgb_valid[features])
    xgb_score = weighted_r2(xgb_valid[TARGET_COL], xgb_pred, xgb_valid[WEIGHT_COL])
    print(f"sample_xgb_valid_weighted_r2={xgb_score:.8f}")

In [ ]:
def walk_forward_lgbm(df, fold_starts, block_days=5, max_train_rows=250_000):
    rows = []
    for fold_start in fold_starts:
        fold_end = fold_start + block_days - 1
        fold_train = df[df[DATE_COL] < fold_start]
        fold_valid = df[(df[DATE_COL] >= fold_start) & (df[DATE_COL] <= fold_end)]
        if len(fold_train) == 0 or len(fold_valid) == 0:
            continue
        fold_train = sample_rows(fold_train, max_train_rows)
        model, score = train_lgbm(fold_train, fold_valid, features, rounds=300)
        rows.append({
            "fold_start": int(fold_start),
            "fold_end": int(fold_end),
            "train_rows": len(fold_train),
            "valid_rows": len(fold_valid),
            "best_iteration": int(model.best_iteration),
            "valid_weighted_r2": score,
        })
        del model
        gc.collect()
    return pd.DataFrame(rows)


wf_df = pd.concat([train_df, valid_df], ignore_index=True).sort_values([DATE_COL, "time_id", "symbol_id"])
valid_dates = sorted(valid_df[DATE_COL].unique())
fold_starts = valid_dates[::10][:2]
walkforward_results = walk_forward_lgbm(wf_df, fold_starts, block_days=5)
walkforward_results

In [ ]:
if (LGBM_MODEL_DIR / "tuning_results.csv").exists():
    display(pd.read_csv(LGBM_MODEL_DIR / "tuning_results.csv"))
else:
    print("No saved LightGBM tuning results found yet.")